[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week9/thinking_demo.ipynb)

# The thinking revolution — hands-on exploration

**PSYC 51.17: Models of language and communication**  
**Week 9 — Companion to Lecture 24**

---

## Learning objectives

By the end of this session, you will:
1. Run an open-weight reasoning model and observe its **visible thinking traces**
2. See how `<think>` tags separate internal reasoning from final answers
3. Compare standard prompting vs. chain-of-thought prompting
4. Build a toy **Mixture of Experts** layer and visualize expert routing
5. Implement a simplified GRPO training loop to see how RL produces reasoning
6. Explore the "budget forcing" trick from the s1 paper

**No API key required** — we use [DeepSeek-R1-Distill-Qwen-7B](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B), an open-weight reasoning model that runs on Colab's free T4 GPU.

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q transformers accelerate torch matplotlib numpy

In [ ]:
import os
import re
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
if torch.cuda.is_available():
    print(f"\u2713 GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("\u26a0 No GPU detected. Parts 1-2 require a GPU runtime.")
    print("  Go to Runtime > Change runtime type > T4 GPU")

print("\u2713 All imports successful!")

## Part 1: Loading an open-weight reasoning model

[DeepSeek-R1](https://arxiv.org/abs/2501.12948) is a reasoning model trained via reinforcement learning (RL) to generate internal thinking traces before answering. The distilled 7B version fits on a free Colab T4 GPU.

Unlike proprietary models (Claude, GPT), DeepSeek-R1's thinking is fully visible — the model outputs `<think>...</think>` tags containing its internal reasoning, followed by the final answer.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

if torch.cuda.is_available():
    print(f"Loading {MODEL_NAME}...")
    print("(This takes ~2 minutes on first run as the model downloads)\n")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model_r1 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    print(f"\u2713 Model loaded! Parameters: {model_r1.num_parameters() / 1e9:.1f}B")
else:
    print("Skipping model load — no GPU available.")

### How thinking tokens work

DeepSeek-R1 generates its response in two phases:
1. **Thinking phase**: The model outputs `<think>` followed by internal reasoning
2. **Answer phase**: After `</think>`, the model outputs its polished final answer

This is the same pattern used by all reasoning models — the key difference is that DeepSeek-R1 is open-weight, so we can see *everything*.

In [ ]:
def generate_with_thinking(question, max_new_tokens=2048):
    """Generate a response from the reasoning model, returning thinking + answer separately."""
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model_r1.device)

    start = time.time()
    with torch.no_grad():
        outputs = model_r1.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
        )
    elapsed = time.time() - start

    # Decode only the new tokens (skip the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    full_response = tokenizer.decode(new_tokens, skip_special_tokens=False)
    num_tokens = len(new_tokens)

    # Parse thinking vs. answer
    # DeepSeek-R1 outputs: <think>...reasoning...</think>

final answer
    # The opening <think> may or may not survive tokenizer decoding
    thinking = ""
    answer = full_response.strip()

    if "</think>" in full_response:
        parts = full_response.split("</think>", 1)
        thinking = parts[0].strip()
        answer = parts[1].strip()
        # Remove leading <think> tag if present
        if thinking.startswith("<think>"):
            thinking = thinking[len("<think>"):].strip()
    elif "<think>" in full_response:
        # Model started thinking but didn't finish (hit token limit)
        thinking = full_response.split("<think>", 1)[1].strip()
        answer = "[generation cut off during thinking]"

    # Clean up special tokens from both fields
    special_tokens = ["<|end_of_sentence|>", "<|begin_of_sentence|>",
                      "<｜end▁of▁sentence｜>", "<｜begin▁of▁sentence｜>",
                      "<｜User｜>", "<｜Assistant｜>"]
    for tok in special_tokens:
        answer = answer.replace(tok, "").strip()
        thinking = thinking.replace(tok, "").strip()

    return {
        "thinking": thinking,
        "answer": answer,
        "full_response": full_response,
        "num_tokens": num_tokens,
        "elapsed": elapsed,
    }


# Test with a math problem from the lecture (the GSM8K-style example)
if torch.cuda.is_available():
    question = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"

    print(f"Question: {question}
")
    print("=" * 70)
    result = generate_with_thinking(question)

    print(f"
--- THINKING ({len(result['thinking'])} chars) ---")
    print(result["thinking"][:800])
    if len(result["thinking"]) > 800:
        print(f"... [{len(result['thinking']) - 800} more chars]")

    print(f"
--- FINAL ANSWER ---")
    print(result["answer"])

    print(f"
--- Stats ---")
    print(f"Total tokens generated: {result['num_tokens']}")
    print(f"Time: {result['elapsed']:.1f}s")
    print(f"Tokens/sec: {result['num_tokens'] / result['elapsed']:.1f}")

### 💡 Discussion

- Look at the thinking trace: can you identify moments of **self-verification**, **backtracking**, or **alternative approaches**?
- These reasoning strategies *emerged from RL training*. The model was never explicitly taught to "check its work" — it learned that checking leads to more correct answers.
- Notice the `<think>...</think>` tags — this is how the model separates internal reasoning from the final answer. Compare this to how Claude uses a `thinking` block and OpenAI hides reasoning entirely.

## Part 2: Comparing reasoning across problem difficulties

Let's test the model on problems of increasing difficulty and observe how the **amount of thinking** scales with problem complexity.

In [ ]:
if torch.cuda.is_available():
    problems = [
        {"label": "Easy (arithmetic)", "q": "What is 15 + 27?", "answer": "42"},
        {"label": "Medium (multi-step)", "q": "A store has 312 apples. They sell 147 in the morning and receive 89 in the afternoon. How many apples remain?", "answer": "254"},
        {"label": "Hard (algebra)", "q": "Find all positive integers n such that n^2 + 2n + 4 is divisible by 7.", "answer": "n = 7k+1 or n = 7k+4"},
    ]

    results_by_difficulty = {}
    for p in problems:
        print(f"\n{'=' * 70}")
        print(f"  {p['label']}")
        print(f"  Q: {p['q']}")
        print(f"{'=' * 70}")

        result = generate_with_thinking(p["q"], max_new_tokens=2048)
        results_by_difficulty[p["label"]] = result

        print(f"\nThinking length: {len(result['thinking'])} chars")
        print(f"Thinking (first 400 chars): {result['thinking'][:400]}")
        if len(result['thinking']) > 400:
            print("...")
        print(f"\nAnswer: {result['answer'][:200]}")
        print(f"Tokens: {result['num_tokens']}, Time: {result['elapsed']:.1f}s")

In [ ]:
if torch.cuda.is_available():
    # Visualize thinking length vs. problem difficulty
    labels = list(results_by_difficulty.keys())
    thinking_lens = [len(results_by_difficulty[l]["thinking"]) for l in labels]
    token_counts = [results_by_difficulty[l]["num_tokens"] for l in labels]
    times = [results_by_difficulty[l]["elapsed"] for l in labels]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].bar(range(len(labels)), thinking_lens, color="#f39c12", alpha=0.8)
    axes[0].set_xticks(range(len(labels)))
    axes[0].set_xticklabels(labels, fontsize=9)
    axes[0].set_title("Thinking Length (chars)", fontsize=13)
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(range(len(labels)), token_counts, color="#0984e3", alpha=0.8)
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_xticklabels(labels, fontsize=9)
    axes[1].set_title("Total Tokens Generated", fontsize=13)
    axes[1].grid(axis="y", alpha=0.3)

    axes[2].bar(range(len(labels)), times, color="#00b894", alpha=0.8)
    axes[2].set_xticks(range(len(labels)))
    axes[2].set_xticklabels(labels, fontsize=9)
    axes[2].set_title("Generation Time (seconds)", fontsize=13)
    axes[2].grid(axis="y", alpha=0.3)

    plt.suptitle("Harder Problems → More Thinking", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

### Examining the full thinking trace

Let's look at the full thinking trace for the hardest problem. Notice the structure: the model breaks the problem down, tries approaches, and sometimes backtracks — all behaviors that emerged from RL training.

In [ ]:
if torch.cuda.is_available():
    hard_result = results_by_difficulty["Hard (algebra)"]
    print(f"Full thinking trace ({len(hard_result['thinking']):,} characters):")
    print("=" * 70)
    print(hard_result["thinking"])
    print("=" * 70)
    print(f"\nFinal answer:")
    print(hard_result["answer"])

### 💡 Discussion

- Does the model think more on harder problems? Look at the bar charts above.
- In the thinking trace, can you identify moments of **self-verification** ("let me check..."), **backtracking** ("actually, I should try..."), or **alternative approaches**?
- These reasoning strategies *emerged from RL training*. The model was never explicitly taught to "check its work."
- How does this compare across providers?

| | OpenAI o-series | Claude | DeepSeek-R1 |
|---|---|---|---|
| Thinking visibility | Hidden (summarized) | **Visible** via API | **Visible** (open-weight) |
| Control | `reasoning_effort` | `budget_tokens` | Token count |
| Thinking tags | Not exposed | `thinking` block | `<think>...</think>` |

## Part 3: Mixture of Experts — sparse activation

Modern frontier models achieve massive scale without proportional compute costs through **Mixture of Experts (MoE)**. Instead of activating every parameter for every token, MoE models route each token to a small subset of "expert" sub-networks.

**Key idea** ([Shazeer et al., 2017](https://arxiv.org/abs/1701.06538)): Replace each dense FFN layer with $N$ smaller expert FFNs. A learned **router** selects the top-$k$ experts per token, and only those experts compute. The result is a weighted combination of expert outputs.

| Model | Total Params | Active/Token | Experts | Routing |
|-------|-------------|-------------|---------|---------|
| [Mixtral 8×7B](https://arxiv.org/abs/2401.04088) | 46.7B | 12.9B | 8 | top-2 |
| [DeepSeek-V3](https://arxiv.org/abs/2412.19437) | 671B | 37B | 256 + 1 shared | top-8 + shared |
| [Llama 4 Maverick](https://ai.meta.com/blog/llama-4-multimodal-intelligence/) | 400B | 17B | 128 + 1 shared | top-1 + shared |

Let's build a toy MoE layer to see how routing works in practice.

In [ ]:
class ToyMoELayer(nn.Module):
    """A toy Mixture of Experts layer with N experts and top-k routing."""
    def __init__(self, input_dim=32, hidden_dim=64, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Router: a simple linear layer that scores each expert
        self.router = nn.Linear(input_dim, num_experts)

        # Expert FFNs: each expert is a small 2-layer network
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
            ) for _ in range(num_experts)
        ])

    def forward(self, x):
        # x shape: (batch_size, input_dim)
        # Step 1: Router scores each expert
        router_logits = self.router(x)  # (batch, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)

        # Step 2: Select top-k experts
        topk_probs, topk_indices = torch.topk(router_probs, self.top_k, dim=-1)

        # Normalize top-k probabilities to sum to 1
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

        # Step 3: Compute weighted sum of selected expert outputs
        output = torch.zeros_like(x)
        for i in range(self.top_k):
            expert_idx = topk_indices[:, i]  # which expert for each sample
            weight = topk_probs[:, i].unsqueeze(-1)  # routing weight

            # Run each selected expert (in practice, this is batched)
            for e_idx in range(self.num_experts):
                mask = (expert_idx == e_idx)
                if mask.any():
                    expert_out = self.experts[e_idx](x[mask])
                    output[mask] += weight[mask] * expert_out

        return output, router_probs, topk_indices


# Create a toy MoE layer and run some inputs through it
torch.manual_seed(42)
moe = ToyMoELayer(input_dim=32, hidden_dim=64, num_experts=8, top_k=2)

# Count parameters
total_params = sum(p.numel() for p in moe.parameters())
single_expert = sum(p.numel() for p in moe.experts[0].parameters())
active_params = single_expert * moe.top_k + sum(p.numel() for p in moe.router.parameters())

print(f"Toy MoE Layer:")
print(f"  Experts: {moe.num_experts}, Top-k: {moe.top_k}")
print(f"  Total parameters:  {total_params:,}")
print(f"  Active per token:  {active_params:,} ({moe.top_k}/{moe.num_experts} experts + router)")
print(f"  Efficiency ratio:  {total_params / active_params:.1f}\u00d7")

# Run 200 random tokens through the layer
x = torch.randn(200, 32)
output, probs, indices = moe(x)
print(f"\nRouted 200 tokens \u2192 output shape: {output.shape}")

In [ ]:
# Visualize expert selection patterns and load balancing
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Expert selection frequency (how often each expert is chosen)
all_selected = indices.flatten().numpy()
expert_counts = np.bincount(all_selected, minlength=moe.num_experts)
colors = ['#00b894' if c > len(x) * moe.top_k / moe.num_experts else '#b2bec3'
          for c in expert_counts]
axes[0].bar(range(moe.num_experts), expert_counts, color=colors, edgecolor='white')
axes[0].axhline(y=len(x) * moe.top_k / moe.num_experts, color='#e17055',
                linestyle='--', label='Ideal (uniform)')
axes[0].set_xlabel('Expert Index')
axes[0].set_ylabel('Times Selected')
axes[0].set_title('Expert Selection Frequency', fontsize=13)
axes[0].legend()
axes[0].set_xticks(range(moe.num_experts))

# 2. Router probability heatmap (first 20 tokens)
im = axes[1].imshow(probs[:20].detach().numpy(), aspect='auto', cmap='YlOrRd')
axes[1].set_xlabel('Expert Index')
axes[1].set_ylabel('Token Index')
axes[1].set_title('Router Probabilities (first 20 tokens)', fontsize=13)
axes[1].set_xticks(range(moe.num_experts))
plt.colorbar(im, ax=axes[1], label='Probability')

# 3. Total vs. active params for real models
models = ['Mixtral\n8\u00d77B', 'DeepSeek\nV3', 'Llama 4\nMaverick']
total = [46.7, 671, 400]
active = [12.9, 37, 17]
x_pos = np.arange(len(models))
width = 0.35
axes[2].bar(x_pos - width/2, total, width, label='Total params (B)',
            color='#6c5ce7', alpha=0.8)
axes[2].bar(x_pos + width/2, active, width, label='Active per token (B)',
            color='#00b894', alpha=0.8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(models, fontsize=10)
axes[2].set_ylabel('Parameters (billions)')
axes[2].set_title('Total vs. Active Parameters', fontsize=13)
axes[2].legend()
axes[2].set_yscale('log')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Mixture of Experts: Sparse Activation in Practice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print load balance stats
print(f"Load balance (ideal = {len(x) * moe.top_k / moe.num_experts:.0f} per expert):")
for i, count in enumerate(expert_counts):
    bar = '\u2588' * (count // 2)
    print(f"  Expert {i}: {count:3d} {bar}")

### 💡 Discussion

- Look at the expert selection frequency: is it balanced? In practice, MoE models use an **auxiliary load-balancing loss** to prevent "expert collapse" (where the router always picks the same few experts).
- The router probability heatmap shows which experts the router *considers* for each token. Note that most probability mass is concentrated on just a few experts — the top-k selection discards the rest.
- Compare the total vs. active parameter counts for real models. DeepSeek-V3 has 671B total parameters but only activates 37B per token — giving it the knowledge capacity of a 671B model at the inference cost of a ~37B model. This is why it could be trained for only ~$5.6M.
- How does MoE interact with reasoning models? A model like DeepSeek-R1 combines MoE (efficient architecture) with RL-trained reasoning (test-time compute). This "both axes" strategy is what makes frontier models increasingly capable *and* affordable.

## Part 4: GRPO — how reasoning is trained

Now let's implement a simplified version of **Group Relative Policy Optimization (GRPO)** — the algorithm DeepSeek used to train R1. We'll train a tiny model to solve simple arithmetic by discovering its own "reasoning" strategies.

### The setup

We'll train a small neural network to solve addition problems. Instead of supervised learning (showing it the steps), we'll use RL: generate multiple candidate answers, check which are correct, and reinforce the patterns that worked.

In [ ]:
class SimpleReasoningModel(nn.Module):
    """A tiny model that learns to solve addition via RL.
    
    Instead of directly predicting the answer, the model generates
    a sequence of 'intermediate tokens' (reasoning steps) before
    producing a final answer. The RL training signal comes only
    from whether the final answer is correct.
    """
    def __init__(self, max_val=20, hidden_dim=128, num_reasoning_steps=5):
        super().__init__()
        self.max_val = max_val
        self.num_reasoning_steps = num_reasoning_steps
        
        # Encode the two input numbers
        self.input_encoder = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Reasoning steps: each step transforms the hidden state
        # This is analogous to generating thinking tokens
        self.reasoning_steps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ) for _ in range(num_reasoning_steps)
        ])
        
        # Gate: decides how much each reasoning step contributes
        # (analogous to the model deciding whether to keep thinking)
        self.gates = nn.ModuleList([
            nn.Linear(hidden_dim, 1) for _ in range(num_reasoning_steps)
        ])
        
        # Output: predict the answer
        self.output_head = nn.Linear(hidden_dim, 2 * max_val + 1)  # possible sums: 0 to 2*max_val
    
    def forward(self, a, b):
        # Encode inputs
        x = torch.stack([a.float(), b.float()], dim=-1)
        h = self.input_encoder(x)
        
        # Apply reasoning steps with gating
        gate_values = []
        for step, gate in zip(self.reasoning_steps, self.gates):
            residual = h
            h_new = step(h)
            g = torch.sigmoid(gate(h))  # how much to use this step
            h = g * h_new + (1 - g) * residual
            gate_values.append(g.mean().item())
        
        # Produce answer logits
        logits = self.output_head(h)
        return logits, gate_values


model = SimpleReasoningModel(max_val=20, hidden_dim=128, num_reasoning_steps=5)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Reasoning steps: {model.num_reasoning_steps}")
print(f"Output classes: {2 * model.max_val + 1} (sums from 0 to {2 * model.max_val})")

In [ ]:
def grpo_train_step(model, batch_size=128, group_size=16, max_val=20, temperature=1.0):
    """One step of simplified GRPO training.
    
    1. Generate random addition problems
    2. For each problem, sample `group_size` candidate answers
    3. Check which are correct
    4. Compute relative advantage (how much better than group average)
    5. Update model to increase probability of above-average solutions
    """
    # Generate random problems: a + b = ?
    a = torch.randint(0, max_val, (batch_size,))
    b = torch.randint(0, max_val, (batch_size,))
    correct = a + b  # ground truth
    
    # Collect all group samples and their rewards
    all_log_probs = []
    all_rewards = []
    
    for _ in range(group_size):
        logits, _ = model(a, b)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Sample an answer from the model's distribution
        probs = F.softmax(logits / temperature, dim=-1)
        sampled = torch.multinomial(probs, 1).squeeze(-1)
        
        selected_log_probs = log_probs.gather(1, sampled.unsqueeze(1)).squeeze(1)
        
        # Reward: +1 if correct, 0 if wrong
        reward = (sampled == correct).float()
        
        all_log_probs.append(selected_log_probs)
        all_rewards.append(reward)
    
    # Stack: shape (group_size, batch_size)
    all_log_probs = torch.stack(all_log_probs)
    all_rewards = torch.stack(all_rewards)
    
    # GRPO: compute group-relative advantage
    # For each problem, compare each sample's reward to the group mean
    group_mean = all_rewards.mean(dim=0, keepdim=True)  # (1, batch_size)
    group_std = all_rewards.std(dim=0, keepdim=True).clamp(min=1e-6)
    advantage = (all_rewards - group_mean) / group_std  # normalized advantage
    
    # Policy gradient: increase log-prob of above-average solutions
    loss = -(advantage.detach() * all_log_probs).mean()
    
    accuracy = all_rewards.mean().item()
    return loss, accuracy


# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
accuracies = []
losses = []

print("Training with GRPO (RL only — no supervised examples)...")
print("The model must DISCOVER how to add numbers through trial and error.")
print(f"Random chance: {1/(2*20+1):.1%} (1 out of {2*20+1} possible sums)\n")

for step in range(1000):
    optimizer.zero_grad()
    loss, acc = grpo_train_step(model, batch_size=128, group_size=16, max_val=20)
    loss.backward()
    optimizer.step()
    
    accuracies.append(acc)
    losses.append(loss.item())
    
    if (step + 1) % 100 == 0:
        recent_acc = np.mean(accuracies[-50:])
        print(f"  Step {step+1:4d}: Loss = {loss.item():.4f}, Accuracy = {recent_acc:.1%}")

print(f"\nFinal accuracy (last 50 steps): {np.mean(accuracies[-50:]):.1%}")

In [ ]:
# Visualize training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Smooth the curves
window = 20
smooth_acc = np.convolve(accuracies, np.ones(window)/window, mode='valid')
smooth_loss = np.convolve(losses, np.ones(window)/window, mode='valid')

ax1.plot(smooth_acc, color="#00b894", linewidth=2)
ax1.set_title("Accuracy (GRPO Training)", fontsize=14)
ax1.set_xlabel("Step")
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1)
ax1.grid(alpha=0.3)
ax1.axhline(y=1/101, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax1.legend()

ax2.plot(smooth_loss, color="#e17055", linewidth=2)
ax2.set_title("Loss (GRPO Training)", fontsize=14)
ax2.set_xlabel("Step")
ax2.set_ylabel("Loss")
ax2.grid(alpha=0.3)

plt.suptitle("Learning to Add via Reinforcement Learning (No Supervised Examples)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Examine the learned gate values — does the model use all reasoning steps?
test_a = torch.randint(0, 20, (100,))
test_b = torch.randint(0, 20, (100,))

with torch.no_grad():
    _, gate_values = model(test_a, test_b)

plt.figure(figsize=(8, 5))
plt.bar(range(1, len(gate_values) + 1), gate_values, color="#6c5ce7", alpha=0.8)
plt.title("Reasoning Step Gate Values (How Much Each Step Contributes)", fontsize=13)
plt.xlabel("Reasoning Step", fontsize=12)
plt.ylabel("Gate Value (0 = skip, 1 = fully use)", fontsize=12)
plt.ylim(0, 1)
plt.xticks(range(1, len(gate_values) + 1))
plt.grid(axis='y', alpha=0.3)
plt.show()

print("Gate values per reasoning step:")
for i, g in enumerate(gate_values):
    bar = '\u2588' * int(g * 30)
    print(f"  Step {i+1}: {g:.3f} {bar}")

### 💡 Discussion

- The model learned to add numbers **without ever being shown how** — it received only a correct/incorrect signal. How does this compare to how DeepSeek-R1-Zero learned to reason?
- Look at the gate values: does the model use all reasoning steps equally, or does it rely more on some than others?
- In the real DeepSeek-R1, emergent behaviors included self-verification and backtracking. Our toy model can't do this (it's a feed-forward network, not autoregressive). What architectural feature of LLMs makes self-correction possible?
- If we increased `num_reasoning_steps` from 5 to 50, would the model get better? When would more steps stop helping?

## Part 5: Comparing reasoning steps to no reasoning

Let's directly compare models with different numbers of reasoning steps to see the effect of "thinking longer."

In [ ]:
# Train models with different numbers of reasoning steps
step_counts = [0, 1, 3, 5, 10]
final_accuracies = {}
training_curves = {}

for n_steps in step_counts:
    print(f"Training model with {n_steps} reasoning steps...")
    m = SimpleReasoningModel(max_val=20, hidden_dim=128, num_reasoning_steps=max(n_steps, 1))
    
    # If 0 steps, disable the gates (force them to 0)
    if n_steps == 0:
        for gate in m.gates:
            gate.weight.data.fill_(-10)  # sigmoid(-10) ≈ 0
            gate.bias.data.fill_(-10)
            gate.weight.requires_grad = False
            gate.bias.requires_grad = False
    
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    accs = []
    
    for step in range(1000):
        opt.zero_grad()
        loss, acc = grpo_train_step(m, batch_size=128, group_size=16, max_val=20)
        loss.backward()
        opt.step()
        accs.append(acc)
    
    final_accuracies[n_steps] = np.mean(accs[-50:])  # average of last 50 steps
    training_curves[n_steps] = accs
    print(f"  Final accuracy: {final_accuracies[n_steps]:.1%}")

print("\n" + "=" * 40)
print("Summary:")
for n, acc in final_accuracies.items():
    print(f"  {n:2d} reasoning steps: {acc:.1%}")

In [ ]:
# Visualize the comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
colors = ["#d63031", "#e17055", "#fdcb6e", "#00b894", "#0984e3"]
for (n_steps, accs), color in zip(training_curves.items(), colors):
    window = 20
    smooth = np.convolve(accs, np.ones(window)/window, mode='valid')
    ax1.plot(smooth, label=f"{n_steps} steps", color=color, linewidth=2)

ax1.set_title("Training Curves by Reasoning Depth", fontsize=14)
ax1.set_xlabel("Training Step")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1)

# Final accuracy bar chart
ax2.bar([str(n) for n in step_counts], 
        [final_accuracies[n] for n in step_counts],
        color=colors, alpha=0.8)
ax2.set_title("Final Accuracy vs. Reasoning Steps", fontsize=14)
ax2.set_xlabel("Number of Reasoning Steps")
ax2.set_ylabel("Accuracy")
ax2.set_ylim(0, 1)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle("More Thinking = Better Performance (up to a point)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 💡 Discussion

- Is the relationship between reasoning steps and accuracy linear? Does it plateau?
- In real reasoning models, the "number of steps" isn't fixed — the model generates tokens until it decides to stop. How might this adaptive behavior be better than a fixed number of steps?
- Our toy model has the same hidden dimension regardless of reasoning steps. In a real transformer, each reasoning token gets the full model's computation. How does this change the scaling dynamics?

## Part 6: Budget forcing — the s1 "Wait" trick

[Muennighoff et al. (2025)](https://arxiv.org/abs/2501.19393) showed that you can force a model to think longer by simply appending "Wait" to its reasoning. Let's test this with our open-weight reasoning model — we'll ask a hard problem and compare what happens when we force additional reflection.

The idea: when the model generates `</think>` (signaling it's done reasoning), we **remove that closing tag and append "Wait"**, forcing the model to continue its internal reasoning before producing a final answer.

In [ ]:
def generate_with_budget_forcing(question, max_new_tokens=4096, num_waits=0):
    """Generate a response, optionally forcing extended thinking with 'Wait' injections.
    
    When num_waits > 0, we generate in stages:
    1. Let the model start thinking normally
    2. When it produces </think>, remove it and append "Wait" to force more reasoning
    3. Repeat for num_waits iterations, then let it finish naturally
    """
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    if num_waits == 0:
        # Standard generation — no budget forcing
        return generate_with_thinking(question, max_new_tokens=max_new_tokens)
    
    # Budget forcing: generate in chunks, injecting "Wait" when model tries to stop thinking
    start = time.time()
    
    # Start with the prompt + <think> tag to ensure thinking mode
    current_text = input_text
    full_generated = ""
    tokens_per_chunk = max_new_tokens // (num_waits + 1)
    waits_done = 0
    
    for chunk_idx in range(num_waits + 1):
        inputs = tokenizer(current_text, return_tensors="pt").to(model_r1.device)
        
        with torch.no_grad():
            outputs = model_r1.generate(
                **inputs,
                max_new_tokens=tokens_per_chunk,
                temperature=0.6,
                top_p=0.95,
                do_sample=True,
            )
        
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        chunk_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        full_generated += chunk_text
        
        # If we still have waits to inject and the model tried to stop thinking
        if waits_done < num_waits and "</think>" in chunk_text:
            # Remove the </think> tag and everything after it, then append "Wait"
            think_end = full_generated.rfind("</think>")
            full_generated = full_generated[:think_end] + "\nWait, let me reconsider.\n"
            current_text = input_text + full_generated
            waits_done += 1
        elif waits_done < num_waits:
            # Model didn't try to stop — just append "Wait" anyway
            full_generated += "\nWait, let me verify this.\n"
            current_text = input_text + full_generated
            waits_done += 1
        else:
            # Final chunk — let it finish naturally
            break
    
    elapsed = time.time() - start
    
    # Parse the final result
    think_match = re.search(r"<think>(.*?)</think>", full_generated, re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        answer = full_generated[think_match.end():].strip()
    else:
        # No closing tag — all text is thinking (model may have run out of tokens)
        thinking = full_generated.replace("<think>", "").strip()
        answer = "(model did not produce a final answer — all tokens used for thinking)"
    
    return {
        "thinking": thinking,
        "answer": answer,
        "full_response": full_generated,
        "num_tokens": len(tokenizer.encode(full_generated)),
        "elapsed": elapsed,
        "waits_injected": waits_done,
    }


# Budget forcing experiment
if torch.cuda.is_available():
    hard_problem = "What is the remainder when 7^100 is divided by 13?"

    print("=" * 70)
    print("BUDGET FORCING EXPERIMENT")
    print("=" * 70)

    print("\n--- Standard (no budget forcing) ---")
    result_standard = generate_with_budget_forcing(hard_problem, max_new_tokens=1024, num_waits=0)
    print(f"Thinking length: {len(result_standard['thinking']):,} chars")
    print(f"Time: {result_standard['elapsed']:.1f}s")
    print(f"Answer: {result_standard['answer'][:300]}")

    print(f"\n--- With 1 'Wait' injection ---")
    result_1wait = generate_with_budget_forcing(hard_problem, max_new_tokens=1536, num_waits=1)
    print(f"Thinking length: {len(result_1wait['thinking']):,} chars")
    print(f"Waits injected: {result_1wait['waits_injected']}")
    print(f"Time: {result_1wait['elapsed']:.1f}s")
    print(f"Answer: {result_1wait['answer'][:300]}")

    print(f"\n--- With 3 'Wait' injections ---")
    result_3wait = generate_with_budget_forcing(hard_problem, max_new_tokens=4096, num_waits=3)
    print(f"Thinking length: {len(result_3wait['thinking']):,} chars")
    print(f"Waits injected: {result_3wait['waits_injected']}")
    print(f"Time: {result_3wait['elapsed']:.1f}s")
    print(f"Answer: {result_3wait['answer'][:300]}")

    print(f"\n--- Comparison ---")
    print(f"{'Condition':<25} {'Thinking (chars)':<20} {'Time (s)':<12}")
    print("-" * 57)
    print(f"{'Standard':<25} {len(result_standard['thinking']):>16,} {result_standard['elapsed']:>8.1f}s")
    print(f"{'1 Wait injection':<25} {len(result_1wait['thinking']):>16,} {result_1wait['elapsed']:>8.1f}s")
    print(f"{'3 Wait injections':<25} {len(result_3wait['thinking']):>16,} {result_3wait['elapsed']:>8.1f}s")
    
    # The correct answer is 7^100 mod 13 = 9 (by Fermat's little theorem: 7^12 ≡ 1 mod 13, 100 = 12*8 + 4, 7^4 = 2401, 2401 mod 13 = 9)
    print(f"\n(Correct answer: 9, by Fermat's little theorem)")

### 💡 Discussion

- Did the "Wait" injections lead to longer thinking traces? Did the model discover errors or try alternative approaches in the extended reasoning?
- The s1 paper found that appending "Wait" improved accuracy by up to 27% on AIME 2024. Why might forcing the model to continue reasoning help — even when the model "thought" it was done?
- This is related to **premature convergence**: the model settles on an answer too quickly. Is this a problem unique to AI, or do humans do this too?
- What are the risks of forcing a model to think longer? Could it *overthink* and make things worse?
- Compare the budget forcing approach here (manipulating the generation stream) with Claude's `budget_tokens` parameter (a first-class API feature). What are the trade-offs?

## Summary

| Concept | What we did | Key insight |
|---------|-------------|-------------|
| **Open-weight reasoning** | Ran DeepSeek-R1-Distill-Qwen-7B on Colab | Reasoning models generate `<think>` traces showing internal reasoning |
| **Thinking vs. difficulty** | Tested easy/medium/hard problems | Harder problems → more thinking tokens (adaptive compute) |
| **Mixture of Experts** | Built a toy MoE layer with 8 experts, top-2 routing | MoE stores more knowledge while keeping per-token compute low |
| **GRPO training** | Implemented simplified RL training | Reasoning strategies emerge from reward signals alone |
| **Reasoning depth** | Compared models with different step counts | More steps help, but with diminishing returns |
| **Budget forcing** | Injected "Wait" to force continued reasoning | Extending thinking can improve accuracy on hard problems |

## Further exploration

1. **Temperature and reasoning**: Try varying the `temperature` parameter in the GRPO training loop. Higher temperature = more exploration. Does this affect what reasoning strategies the model discovers?

2. **Problem difficulty**: Increase `max_val` in the GRPO experiment. At what point does the model need more reasoning steps to maintain accuracy? Is there a predictable relationship?

3. **Transfer**: Train the GRPO model on addition, then test it on subtraction (without retraining). Does the "reasoning" transfer, or is it task-specific?

4. **Budget forcing depth**: Try increasing `num_waits` in the budget forcing experiment (e.g., 5 or 10 injections). Is there a point of diminishing returns — or does the model start *overthinking* and get worse?

5. **Compare open models**: Try loading a different open reasoning model, such as [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) (which uses `/think` and `/no_think` directives instead of `<think>` tags). How do the thinking traces compare?

6. **Thinking token efficiency**: For the difficulty comparison (Part 2), compute the ratio of thinking tokens to answer tokens. Do harder problems have a higher ratio? Is there an "optimal" ratio?